# 05 — External Dataset Feature Importance

Explores two public datasets to understand which behavioral signals matter for player/bot detection:

| Dataset | Type | Target |
|---|---|---|
| **CS2CD** | CS2 match-level statistics | Win/loss prediction |
| **CaptchaSolve30k** | Mouse trajectories during CAPTCHA solving | Bot vs human |

Goal: identify which features overlap with BehaviorDNA's 18 behavioral signals and which are complementary.

In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from lightgbm import LGBMClassifier
from sklearn.preprocessing import LabelEncoder

ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))

from pipeline.features.run import FEATURE_COLS

sns.set_theme(style="darkgrid", palette="muted")
plt.rcParams["figure.dpi"] = 120

EXT_DIR = ROOT / "data" / "external"
CS2CD_DIR = EXT_DIR / "cs2cd"
CAPTCHA_DIR = EXT_DIR / "captcha30k"

## Dataset Downloads

Requires a Kaggle API token at `~/.kaggle/kaggle.json`.
Find your token at [kaggle.com/settings/account](https://www.kaggle.com/settings/account) → API → Create New Token.

**Verify the exact dataset slugs** before running — update the commands below if needed:

```bash
# CS2CD — Counter-Strike 2 match statistics
!kaggle datasets download -d <cs2cd-slug> -p data/external/cs2cd --unzip

# CaptchaSolve30k — mouse movement + bot/human annotations
!kaggle datasets download -d <captcha30k-slug> -p data/external/captcha30k --unzip
```

Search on Kaggle: https://www.kaggle.com/search?q=cs2+match+statistics
and: https://www.kaggle.com/search?q=captcha+mouse+movement+bot

---
## Section A — CS2CD: Match-Level Features

In [ ]:
# Auto-detect the CSV file inside the download directory
cs2cd_csvs = list(CS2CD_DIR.glob("**/*.csv"))
if not cs2cd_csvs:
    raise FileNotFoundError(
        f"No CSV found in {CS2CD_DIR}. Run the kaggle download cell first."
    )
print("Found:", cs2cd_csvs)
cs2 = pd.read_csv(cs2cd_csvs[0])
print(f"Shape: {cs2.shape}")
cs2.head()

In [ ]:
print(cs2.dtypes)
print("\nNaN rates:")
print((cs2.isna().mean() * 100).sort_values(ascending=False).head(20))

In [ ]:
cs2.describe().T

In [ ]:
# Correlation heatmap — numeric columns only
num_cols = cs2.select_dtypes(include="number").columns.tolist()
corr = cs2[num_cols].corr()

fig, ax = plt.subplots(figsize=(min(20, len(num_cols)), min(16, len(num_cols))))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=len(num_cols) <= 20, fmt=".2f",
            cmap="coolwarm", center=0, ax=ax, linewidths=0.5)
ax.set_title("CS2CD — Feature Correlation Matrix")
plt.tight_layout()
plt.show()

In [ ]:
# Identify the target column — adapt to actual CS2CD column names
# Common candidates: 'result', 'win', 'outcome', 'winner'
target_candidates = [c for c in cs2.columns if c.lower() in
                     ("result", "win", "outcome", "winner", "won", "match_result")]
print("Target candidates:", target_candidates)
print("\nAll columns:", cs2.columns.tolist())

In [ ]:
# --- Update TARGET_COL to match the actual column name in the dataset ---
TARGET_COL = target_candidates[0] if target_candidates else None

if TARGET_COL is None:
    print("No target column found — skipping LightGBM training. Set TARGET_COL manually.")
else:
    feature_cols = [c for c in num_cols if c != TARGET_COL]
    X = cs2[feature_cols].fillna(0)
    y_raw = cs2[TARGET_COL]

    if y_raw.dtype == object:
        le = LabelEncoder()
        y = le.fit_transform(y_raw)
        print("Classes:", le.classes_)
    else:
        y = y_raw.values

    lgbm = LGBMClassifier(n_estimators=200, num_leaves=31, verbose=-1,
                           class_weight="balanced")
    lgbm.fit(X, y)

    importances = pd.Series(lgbm.feature_importances_, index=feature_cols)
    top20 = importances.nlargest(20).sort_values()

    fig, ax = plt.subplots(figsize=(10, 7))
    top20.plot.barh(ax=ax, color="steelblue")
    ax.set_title(f"CS2CD — Top 20 Feature Importances (target: {TARGET_COL})")
    ax.set_xlabel("LightGBM gain importance")
    plt.tight_layout()
    plt.show()

    # Highlight features that appear in BehaviorDNA
    overlap = set(importances.nlargest(20).index) & set(FEATURE_COLS)
    print(f"\nOverlap with BehaviorDNA FEATURE_COLS in top-20: {overlap or 'none'}")

---
## Section B — CaptchaSolve30k: Mouse Trajectories + Bot Labels

In [ ]:
captcha_files = list(CAPTCHA_DIR.glob("**/*.csv")) + list(CAPTCHA_DIR.glob("**/*.parquet"))
if not captcha_files:
    raise FileNotFoundError(
        f"No data found in {CAPTCHA_DIR}. Run the kaggle download cell first."
    )
print("Found:", captcha_files)

cap_path = captcha_files[0]
cap = pd.read_parquet(cap_path) if cap_path.suffix == ".parquet" else pd.read_csv(cap_path)
print(f"Shape: {cap.shape}")
cap.head()

In [ ]:
print(cap.dtypes)
print("\nNaN rates:")
print((cap.isna().mean() * 100).sort_values(ascending=False).head(20))

# Identify label and movement columns
label_candidates = [c for c in cap.columns if c.lower() in
                    ("label", "is_bot", "bot", "type", "class", "human")]
print("\nLabel candidates:", label_candidates)
print("All columns:", cap.columns.tolist())

In [ ]:
# --- Update LABEL_COL to match the actual column name ---
LABEL_COL = label_candidates[0] if label_candidates else None

if LABEL_COL:
    print(f"Label distribution ('{LABEL_COL}'):")
    print(cap[LABEL_COL].value_counts())

    fig, ax = plt.subplots(figsize=(5, 3))
    cap[LABEL_COL].value_counts().plot.bar(ax=ax, color=["steelblue", "tomato"])
    ax.set_title("CaptchaSolve30k — Label Distribution")
    ax.set_xlabel(LABEL_COL)
    ax.set_ylabel("count")
    plt.tight_layout()
    plt.show()

In [ ]:
# Distribution comparison: bot vs human across numeric features
cap_num = cap.select_dtypes(include="number")
plot_cols = cap_num.columns[:12] if len(cap_num.columns) > 12 else cap_num.columns

if LABEL_COL and LABEL_COL in cap.columns:
    n = len(plot_cols)
    cols = 3
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(15, rows * 3))
    axes = axes.flatten()

    for i, col in enumerate(plot_cols):
        for label_val, color in zip(cap[LABEL_COL].unique(), ["steelblue", "tomato"]):
            subset = cap[cap[LABEL_COL] == label_val][col].dropna()
            axes[i].hist(subset, bins=40, alpha=0.5, color=color,
                         label=str(label_val), density=True)
        axes[i].set_title(col, fontsize=9)
        axes[i].legend(fontsize=7)

    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    fig.suptitle("CaptchaSolve30k — Feature Distributions (bot vs human)", fontsize=13)
    plt.tight_layout()
    plt.show()

In [ ]:
# Train binary classifier and extract feature importances
if LABEL_COL:
    cap_feat_cols = [c for c in cap_num.columns if c != LABEL_COL]
    X_cap = cap[cap_feat_cols].fillna(0)
    y_cap_raw = cap[LABEL_COL]

    le_cap = LabelEncoder()
    y_cap = le_cap.fit_transform(y_cap_raw)

    lgbm_cap = LGBMClassifier(n_estimators=200, num_leaves=31, verbose=-1,
                               class_weight="balanced")
    lgbm_cap.fit(X_cap, y_cap)

    imp_cap = pd.Series(lgbm_cap.feature_importances_, index=cap_feat_cols)
    top20_cap = imp_cap.nlargest(20).sort_values()

    # Colour bars that overlap with BehaviorDNA FEATURE_COLS
    colors = ["tomato" if f in FEATURE_COLS else "steelblue" for f in top20_cap.index]

    fig, ax = plt.subplots(figsize=(10, 7))
    top20_cap.plot.barh(ax=ax, color=colors)
    ax.set_title("CaptchaSolve30k — Top 20 Feature Importances\n(red = overlaps with BehaviorDNA)")
    ax.set_xlabel("LightGBM gain importance")
    plt.tight_layout()
    plt.show()

    overlap_cap = set(top20_cap.index) & set(FEATURE_COLS)
    print(f"BehaviorDNA features in CaptchaSolve30k top-20: {overlap_cap or 'none (column names differ)'}")

---
## Section C — Cross-Dataset Summary

In [ ]:
# Build summary table: feature / in BehaviorDNA / top-20 CS2CD / top-20 CaptchaSolve30k
all_features = set(FEATURE_COLS)

try:
    top20_cs2_set = set(importances.nlargest(20).index)
except NameError:
    top20_cs2_set = set()

try:
    top20_cap_set = set(imp_cap.nlargest(20).index)
except NameError:
    top20_cap_set = set()

all_keys = all_features | top20_cs2_set | top20_cap_set
rows = []
for f in sorted(all_keys):
    rows.append({
        "feature": f,
        "BehaviorDNA": "✓" if f in all_features else "",
        "CS2CD top-20": "✓" if f in top20_cs2_set else "",
        "CaptchaSolve30k top-20": "✓" if f in top20_cap_set else "",
    })

summary = pd.DataFrame(rows).set_index("feature")
summary

### Key Takeaways

*(Fill in after running the analysis)*

- **CS2CD** is match-level data (kills, deaths, rounds won) — minimal overlap with per-event mouse/keyboard signals. Useful for match outcome prediction but not directly applicable to behavioral biometrics.
- **CaptchaSolve30k** is the closest public analogue to BehaviorDNA's approach — mouse trajectory signals during a constrained task. High overlap expected with speed, jitter, and click interval features.
- BehaviorDNA's `wasd_rhythm` and `burst_rate` are game-specific and unlikely to appear in CAPTCHA data.